# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print("Description:")
print(metadata.description)
print("\nSuggested Citation:")
print(metadata.cite_as)
print("\nKeywords:")
print(metadata.keywords)

## 2. Data Overview
Review available record sets, fields, column `@id`s, and reference IDs.

In [ ]:
# Fetch the record set metadata using `@id`
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
if record_sets:
    for rset in record_sets:
        print(f"Record Set: {rset.name} | @id: {rset.id}")
        print("  Fields:")
        for field in rset.fields:
            print(f"    - {field.name} | @id: {field.id}")
        if hasattr(rset, 'columns') and rset.columns:
            print("  Columns:")
            for col in rset.columns:
                print(f"    - {col.name} | @id: {col.id}")
        print()
else:
    print('No record sets found. Please check the dataset schema.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids (for demo, we'll use the first if available)
record_set_ids = [rset.id for rset in record_sets]
dataframes = {}

if len(record_set_ids) == 0:
    print("No record sets are defined in this schema.\nCheck the metadata's 'recordSet' or 'distribution' for files or tables.")
else:
    for record_set_id in record_set_ids:
        # Each record is a dict; columns/fields as keys referenced by their `@id`
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set '{record_set_id}'")

    # Show columns for the first loaded DataFrame
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All fields referenced use their `@id`.

In [ ]:
# Select a record set and fields by @id
if len(record_set_ids) == 0:
    print('No record sets available for EDA.')
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Show columns to help pick a numeric field (by @id)
    print(f"Available fields (@id) in {record_set_id}:\n", df.columns.tolist())

    # Attempt to auto-select a numeric field for demonstration
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if not numeric_field:
        print('No numeric field found in the first record set. Skipping EDA.')
    else:
        threshold = df[numeric_field].mean()  # Example: use mean as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (if exists, by @id)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and not df[col].isnull().all():
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}: (showing mean of groups)")
            print(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if len(record_set_ids) == 0 or not numeric_field:
    print('No data available for visualization.')
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot vs group field (if group_field found)
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We showcased how to access Croissant datasets with `mlcroissant`, always using entity `@id`s for reference.
- All exploration, filtering, and grouping were driven by the schema structure definitions and referenced with `@id`.
- Continue with domain-specific analytics by selecting the fields and relationships as needed for your FAIR^2 dataset use case.